# Loss Leaders or Dead Weight?
## Basket-Aware Profitability and Recommendations for a Taiwanese Supermarket

## Executive Summary

Supermarkets usually judge a product by its own margin, and items that sell below cost are the first to be cut or repriced. That view ignores the rest of the basket. A product that loses money on its own can still be worth keeping if the shoppers who buy it also fill their baskets with profitable items. Promotions have the same blind spot: every shopper gets the same flyer, regardless of what they actually buy.

Using four months of purchase lines from the Ta Feng supermarket (November 2000 to February 2001), this report combines **frequent itemset mining (FIM)** and **collaborative filtering (CF)**, with profit running through both:

1. **FP-Growth** finds which product subclasses are bought together.
2. A new measure, **basket-adjusted profit (BAP)**, adds to each subclass's own margin the *basket halo*: how much more the rest of its baskets earn than baskets of the same size. Own margin and halo sort every subclass into **stars, loss leaders, passengers and dead weight**.
3. A **recommender** (popularity, repeat-buy, item-based kNN and implicit ALS) predicts what each customer buys next, and a **profit-aware re-ranking** step trades a controlled amount of accuracy for profit. A margin-only re-ranking serves as the control, to test whether the basket adjustment adds anything.
4. Everything is evaluated on time splits: λ is tuned on November–December → January (with a check for Chinese New Year), then the model is retrained on November–January and tested on February, with profit measured from **actual February purchases**, not from BAP.

The headline numbers depend on the run and are printed in **Results at a Glance** near the end, which is generated from the results when the notebook runs. The report ends with an action list for the store: which loss-making items to keep, which to drop, what to bundle, and which offers to send each customer.

## Problem Statement

Retail category reviews rank products by their own margin. This hides two things:

* **Cross-subsidy.** A loss leader may pull profitable purchases into the basket, so cutting it can lose more than it saves.
* **One-size promotions.** Offers are the same for every shopper and are chosen without looking at what each customer is likely to buy next.

We ask two questions:

1. **Which products earn their place through the baskets they bring in, and which are dead weight?**
2. **Can we recommend what each customer buys next, in a way that also favors profitable items, and does a basket-aware notion of profit do better than plain margin?**

## Motivation

Ta Feng is one of the few public grocery datasets that records **cost** as well as **price** on every line, so profit can be computed at the line, item, basket and customer level. Most market-basket studies stop at support, confidence and lift; most recommender studies stop at accuracy. Neither tells a store manager whether a recommendation or a shelf decision makes money. Linking the two through profit turns the classic market-basket question ("what is bought together?") into a business one ("what is worth keeping, bundling and promoting?").

BAP measures **association, not causation**: a loss leader with a large halo is *bought alongside* profitable items, which is necessary but not sufficient for it to *cause* those purchases. We return to this in the limitations.

## Data Source and Description

The data is the **Ta Feng Grocery Dataset** read directly from the shared folder on Jojie (`/mnt/data/public/tafeng`, four monthly files `D11`, `D12`, `D01`, `D02`; the same data is published on Kaggle as `ta_feng_all_months_merged.csv`); no raw or processed data is copied into this submission. It holds 817,741 purchase lines from a Taiwanese supermarket between November 2000 and February 2001. The original columns are `TRANSACTION_DT`, `CUSTOMER_ID`, `AGE_GROUP`, `PIN_CODE`, `PRODUCT_SUBCLASS`, `PRODUCT_ID`, `AMOUNT`, `ASSET` and `SALES_PRICE`, renamed as follows.

| Column (← original) | Description |
|---|---|
| `date` ← `TRANSACTION_DT` | Transaction date |
| `customer` ← `CUSTOMER_ID` | Customer ID (for purchase tracking) |
| `age` ← `AGE_GROUP` | Age group, coded A–K in the monthly files: A <25, B 25–29, …, I 60–64, J >64, K unknown |
| `area` ← `PIN_CODE` | Residence area, coded A–H: A–F are six zip-code areas, G other, H unknown |
| `subclass` ← `PRODUCT_SUBCLASS` | Product subclass (~2,000 categories); this is the item level used throughout |
| `product` ← `PRODUCT_ID` | Product ID (barcode level) |
| `qty` ← `AMOUNT` | Quantity bought |
| `cost` ← `ASSET` | Store cost |
| `price` ← `SALES_PRICE` | Sales price |

There are no explicit ratings and no product names, so items are referred to by their subclass code.

### Supplementary code

All processing, mining, modelling and plotting code lives in the `src/` folder and is imported below. This notebook only calls those functions and discusses the results.

| File | Contents |
|---|---|
| `src/data_prep.py` | loading both Ta Feng layouts, data-quality checks, cleaning, baskets, time windows |
| `src/basket_mining.py` | FP-Growth (pyFIM, with a pure-Python fallback), association rules, maximal itemsets, BAP, pull-through, bundles, segment rules |
| `src/recommenders.py` | utility matrix, popularity, repeat-buy, item-based kNN, implicit ALS |
| `src/evaluation.py` | time-split test sets, precision/recall/NDCG@10, profit capture, profit-aware re-ranking, λ selection |
| `src/plots.py` | every figure (also saved as PNG in `figures/`) |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import data_prep as dp
from src import basket_mining as bm
from src import recommenders as rc
from src import evaluation as ev
from src import plots

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def say(text):
    # results-driven narrative, so the text always matches the numbers
    display(Markdown(text))

### Settings

`DATA_DIR` points to the shared Ta Feng folder on Jojie, which holds the four monthly files `D11`, `D12`, `D01` and `D02` (November 2000 to February 2001). The loader also accepts the merged Kaggle release: set `DATA_DIR` to the folder or straight to `ta_feng_all_months_merged.csv`. If `DATA_DIR` does not exist, the loader searches `/mnt/data/public/` for a Ta Feng folder. No path contains a user name.

In [ ]:
DATA_DIR = "/mnt/data/public/tafeng"   # shared Ta Feng folder on Jojie (D11, D12, D01, D02)

MIN_SUPPORT = 0.003         # FP-Growth relative minimum support (0.3% of baskets)
MAX_ITEMSET = 3             # largest itemset mined
MIN_BAP_BASKETS = 30        # subclasses need >= 30 baskets for a BAP score
MIN_TRAIN_BASKETS = 2       # customers need >= 2 training baskets to be evaluated
SEGMENT_SUPPORT = 0.005     # relative support inside each segment
SEED = 0

# validation (tune) and test (report) windows
VAL_TRAIN_END, VAL_TEST_END = "2001-01-01", "2001-02-01"
TEST_TRAIN_END, TEST_END = "2001-02-01", "2001-03-01"
# January split used for the Chinese New Year check (CNY = 24 Jan 2001)
CNY_WINDOW_START = "2001-01-14"

print("pyFIM available:", bm.HAS_FIM)

## Data Loading and Quality Checks

The game plan named four things to check before any modelling: zero or negative costs, returns, sales below cost, and whether `price` and `cost` are **line totals** or unit values. The last one matters: if they are line totals, profit is simply `price − cost`; if they are unit values, both must be multiplied by quantity first.

In [ ]:
raw = dp.load_tafeng(DATA_DIR)
print(f"{len(raw):,} lines | {raw['customer'].nunique():,} customers | "
      f"{raw['subclass'].nunique():,} subclasses | "
      f"{raw['product'].nunique():,} products | "
      f"{raw['date'].min():%Y-%m-%d} to {raw['date'].max():%Y-%m-%d}")
raw.head()

In [ ]:
quality = dp.quality_report(raw)
quality

**Line totals or unit values?** Within one product the unit price barely moves, so if `price` is a line total it should rise with `qty` (correlation near 1) while `price / qty` stays flat (low coefficient of variation). If `price` were a unit value, the reverse would hold.

In [ ]:
lt = dp.line_total_check(raw)
display(lt)

is_total = (lt.loc["price", "median CV of value / qty"]
            < lt.loc["price", "median CV of raw value"]) and \
           (lt.loc["price", "median corr(qty, raw value)"] > 0.5)
say(f"Across {lt.attrs['products_used']:,} products with varying quantities, "
    f"`price / qty` varies far less than `price` "
    f"(median CV {lt.loc['price', 'median CV of value / qty']:.2f} vs "
    f"{lt.loc['price', 'median CV of raw value']:.2f}) and `price` tracks `qty` "
    f"(median correlation {lt.loc['price', 'median corr(qty, raw value)']:.2f}); "
    f"the same holds for `cost`. "
    + ("**Price and cost are line totals**, so line profit is `price − cost` "
       "without multiplying by quantity."
       if is_total else
       "**The test does not support line totals**; check this before "
       "trusting the profit figures."))

In [ ]:
lines = dp.clean(raw)
dropped = len(raw) - len(lines)
below_cost = (lines["profit"] < 0).mean()
caps = lines.attrs["caps"]
say(f"Cleaning removes **{dropped:,} lines ({dropped / len(raw):.2%})**: lines "
    f"with non-positive quantity, cost or price, a cost more than five times "
    f"the price, or a bulk quantity or price above the 99.95th percentile "
    f"(qty > {caps['qty']:,.0f} or price > NT${caps['price']:,.0f}), since a "
    f"few wholesale-sized lines would otherwise dominate the per-basket "
    f"averages. Lines sold below cost are kept on purpose: they are "
    f"**{below_cost:.1%}** of the clean lines and are exactly what this report "
    f"is about. Total clean revenue is NT${lines['price'].sum():,.0f} and "
    f"total profit NT${lines['profit'].sum():,.0f} "
    f"({lines['profit'].sum() / lines['price'].sum():.1%} margin).")

In [ ]:
plots.daily_sales(lines);

Sales follow a weekly rhythm and build up in the run-up to **Chinese New Year (24 January 2001)**, then drop sharply over the holiday. This seasonal spike is why the evaluation checks whether January, the validation month, distorts the tuning (see *Does Chinese New Year distort the tuning?*).

## Building Baskets

**One basket is one customer on one day**, at the product-subclass level. Using subclasses (~2,000) instead of barcodes (~24,000) pools different brands and pack sizes of the same thing, which gives the mining and the recommender enough data per item.

The store's insight tables (FIM, BAP) use the training months November–January, so that February stays an untouched holdout for both the recommender and the BAP persistence check.

In [ ]:
train_lines = dp.period(lines, end=TEST_TRAIN_END)      # Nov-Jan
hold_lines = dp.period(lines, start=TEST_TRAIN_END)     # Feb

item_basket, baskets = dp.make_baskets(train_lines)
db = dp.transactions(item_basket)
n_baskets = len(db)

summary = pd.Series({
    "baskets (Nov-Jan)": n_baskets,
    "customers": baskets["customer"].nunique(),
    "subclasses": item_basket["subclass"].nunique(),
    "mean subclasses per basket": baskets["size"].mean(),
    "median subclasses per basket": baskets["size"].median(),
    "mean basket profit (NT$)": baskets["profit"].mean(),
    "baskets per customer": n_baskets / baskets["customer"].nunique(),
})
summary.to_frame("value")

## Frequent Itemset Mining and Association Rules

### Method

As in class, the **support** of an itemset $X$ is the number of baskets containing it, and its relative support is $relSup(X) = sup(X)/N$. For a rule $A \rightarrow B$,

$$
conf(A \rightarrow B) = \frac{sup(A \cup B)}{sup(A)}, \qquad
lift(A \rightarrow B) = \frac{conf(A \rightarrow B)}{relSup(B)}.
$$

Itemsets are mined with **FP-Growth** (`fim.fpgrowth`, as in class), since with ~2,000 subclasses Apriori's candidate generation would explode. Rules with one consequent are then generated from the frequent itemsets with the class `assoc` logic. The minimum support is 0.3% of baskets: low enough to reach pairs involving mid-selling subclasses, high enough that each rule rests on hundreds of baskets.

In [ ]:
min_count = int(np.ceil(MIN_SUPPORT * n_baskets))
supports = bm.frequent_itemsets(db, min_count, zmax=MAX_ITEMSET)
rules = bm.assoc(supports, n_baskets)

by_size = pd.Series([len(k) for k in supports]).value_counts().sort_index()
print(f"min support = {min_count} baskets ({MIN_SUPPORT:.1%})")
print("frequent itemsets by size:", by_size.to_dict())
print(f"rules: {len(rules):,}")

In [ ]:
# most common pairs (support) and strongest rules (lift)
pairs = (rules[rules["n_ante"] == 1]
         .assign(pair=lambda d: [tuple(sorted((a[0], c)))
                                 for a, c in zip(d["antecedent"], d["consequent"])])
         .drop_duplicates("pair"))
display(pairs.nlargest(10, "support")[
    ["antecedent", "consequent", "support", "rel_support", "lift"]])
display(rules.head(10)[
    ["antecedent", "consequent", "support", "confidence", "lift"]])

In [ ]:
max_fi = bm.maximal(supports)
top_pair = pairs.nlargest(1, "support").iloc[0]
top_rule = rules.iloc[0]
say(f"FP-Growth finds **{len(supports):,} frequent itemsets** "
    f"({by_size.get(2, 0):,} pairs, {by_size.get(3, 0):,} triples) and "
    f"**{len(max_fi):,} maximal itemsets** of two or more subclasses. "
    f"The most common pair is {top_pair['pair']} "
    f"({top_pair['rel_support']:.1%} of baskets, lift {top_pair['lift']:.2f}). "
    f"The strongest rule is {top_rule['antecedent']} → {top_rule['consequent']} "
    f"(confidence {top_rule['confidence']:.2f}, lift {top_rule['lift']:.2f}). "
    f"Confidences are modest, as is typical of grocery data: even strong "
    f"complements appear together in only a fraction of baskets, which is why "
    f"lift rather than confidence is used to rank rules.")

Support and lift alone say nothing about money. The next section attaches profit to every subclass and then uses these rules to explain *where* a loss leader's halo comes from.

## Basket-Adjusted Profit

### Method

For subclass $i$ in basket $b$, let $p_{ib}$ be the profit from $i$ itself, $P_b$ the profit of the whole basket, and $n_b$ the number of distinct subclasses in it. Then

* **own margin** $m_i = \frac{1}{|B_i|}\sum_{b \in B_i} p_{ib}$, the average profit per basket from the item itself, where $B_i$ is the set of baskets containing $i$;
* the other items in $b$ earn $o_{ib} = P_b - p_{ib}$.

Bigger baskets earn more, and every item lands in bigger-than-average baskets simply because big baskets contain more items. To remove that effect, the halo compares $o_{ib}$ with **baskets of the same size**. Taking a random item out of a basket of size $n$ leaves, on average, $P_b\,(n-1)/n$ for the others, so the size baseline is

$$
E_n = \operatorname{mean}_{b:\,n_b = n} \; P_b\,\frac{n_b-1}{n_b},
$$

and

$$
\textbf{halo}\;\; h_i = \frac{1}{|B_i|}\sum_{b \in B_i} \left(o_{ib} - E_{n_b}\right), \qquad
\textbf{BAP}_i = m_i + h_i.
$$

A useful property of this baseline: within each basket size the residuals $o_{ib}-E_{n_b}$ sum to zero over all items, so a positive halo means *better than an average item in a basket that size*, not just "bought in big baskets". Basket sizes above 25 are pooled. Only subclasses in at least 30 training baskets are scored, and every halo comes with a standard error, so we can tell a real halo from noise.

With $m_i > 0$ and $h_i$ above the median halo, subclasses fall into four groups:

| | **halo above median** | **halo at or below median** |
|---|---|---|
| **own margin > 0** | **Star**: profitable and lifts the basket | **Passenger**: profitable, adds nothing beyond itself |
| **own margin ≤ 0** | **Loss leader**: loses money but sits in rich baskets | **Dead weight**: loses money and lifts nothing |

In [ ]:
bap = bm.basket_adjusted_profit(item_basket, baskets,
                                min_baskets=MIN_BAP_BASKETS)
plots.basket_baseline(baskets, bap.attrs["expected_other"]);

In [ ]:
quad = bm.quadrant_summary(bap)
display(quad)
plots.quadrant_map(bap);

In [ ]:
lead = bap[bap["quadrant"] == "Loss leader"]
dead = bap[bap["quadrant"] == "Dead weight"]
loss_items = bap[bap["own_margin"] <= 0]
say(f"**{len(bap):,} subclasses** have enough baskets for a BAP score. "
    f"{len(loss_items):,} of them lose money on their own. By own margin alone, "
    f"all {len(loss_items):,} would be candidates for cutting or repricing. "
    f"The halo splits them: **{len(lead):,} are loss leaders** "
    f"(revenue share {quad.loc['Loss leader', 'revenue_share']:.1%}) and "
    f"**{len(dead):,} are dead weight** "
    f"(revenue share {quad.loc['Dead weight', 'revenue_share']:.1%}). "
    f"Of the loss leaders, **{(lead['bap'] > 0).sum():,} have a positive BAP**, "
    f"meaning the rest of their baskets earns more than their own loss, and "
    f"{(lead['halo_t'] >= 2).sum():,} have a halo at least two standard errors "
    f"above zero.")

In [ ]:
cols = ["n_baskets", "own_margin", "margin_rate", "halo", "halo_se",
        "halo_t", "bap"]
print("Loss leaders with the largest BAP")
display(lead.sort_values("bap", ascending=False)[cols].head(10))
print("Dead weight with the most negative BAP")
display(dead.sort_values("bap")[cols].head(10))

### Does the halo persist out of time?

BAP was computed on November–January. If the halo were noise it would not show up again in February. We recompute BAP on the February baskets alone and compare the rankings.

In [ ]:
hold_ib, hold_bk = dp.make_baskets(hold_lines)
bap_hold = bm.basket_adjusted_profit(hold_ib, hold_bk,
                                     min_baskets=MIN_BAP_BASKETS)
persist = bm.bap_persistence(bap, bap_hold)
display(persist.to_frame("Nov-Jan vs Feb"))
plots.bap_persistence(bap, bap_hold);

In [ ]:
say(f"On the {int(persist['subclasses']):,} subclasses scored in both periods, "
    f"the Spearman correlation between training and February is "
    f"{persist['own_margin']:.2f} for own margin, "
    f"**{persist['halo']:.2f} for the halo** and {persist['bap']:.2f} for BAP, "
    f"and {persist['same_quadrant']:.0%} of subclasses stay in the same "
    f"quadrant. "
    + ("The halo is a stable property of a subclass, not a one-month artefact, "
       "so it is safe to act on." if persist["halo"] >= 0.5 else
       "The halo is only moderately stable month to month, so decisions "
       "should favour subclasses whose halo is also statistically strong."))

### Where does a loss leader's halo come from?

FP-Growth rules show *which* profitable subclasses each loss leader pulls in. For a rule $\{\text{leader}\} \rightarrow j$ with $j$ profitable, the extra profit per leader basket compared with an average basket is

$$
\Delta_j = \big(conf(\text{leader} \rightarrow j) - relSup(j)\big)\, m_j .
$$

In [ ]:
pull = bm.pull_through(rules, bap)
pull.head(15)

In [ ]:
if len(pull):
    by_leader = pull.groupby("leader")["extra_profit"].sum()
    covered = by_leader.reindex(lead.index).fillna(0)
    say(f"{(covered > 0).sum():,} of the {len(lead):,} loss leaders have at "
        f"least one frequent rule pointing to a profitable subclass. "
        f"For the remaining ones the halo is spread across many items, none "
        f"frequent enough on its own to pass the support threshold.")
else:
    say("No loss leader has a frequent pair rule at this support threshold; "
        "their halos are spread across many low-support items.")

### Bundles and shelf placement

A bundle should be strongly associated, profitable and common, so pairs with lift ≥ 1.5 and a positive combined margin are ranked by $lift \times (m_a + m_b) \times sup$.

In [ ]:
bundles = bm.bundle_pairs(rules, bap, min_lift=1.5)
bundles.head(10)

## Recommender System

### Implicit utility matrix

There are no ratings, so the utility matrix is built from behaviour. The implicit rating of customer $u$ for subclass $i$ is

$$
r_{ui} = \log\big(1 + \text{number of distinct days } u \text{ bought } i\big).
$$

Counting **days** rather than units means that a family buying a 12-pack does not look twelve times as keen as someone buying one bottle, and the log damps heavy repeat buyers. Unlike the explicit ratings in class (Jester, sushi), an empty cell here means "did not buy", not "unknown".

### Models

* **Popularity**: the same list for everyone, ranked by number of buyers.
* **Repeat-buy**: the customer's own $r_{ui}$. Not collaborative, but grocery shoppers repeat heavily, so any CF model has to be judged against it.
* **Item-based kNN** (class neighbourhood CF): cosine similarity between item columns, keeping each item's $k$ nearest neighbours, with
$$
\hat r_{ui} = \sum_{j \in N_k(i) \cap I_u} sim(i,j)\, r_{uj}.
$$
The class formula divides by $\sum sim(i,j)$. With positive-only data that division turns the score into an average of the customer's ratings and ranks highest the items backed by a single weak neighbour, so the plain sum (total evidence) is used instead. For the same reason the matrix is not mean-centred.
* **Implicit ALS** (class latent-factor CF, adapted to implicit feedback; Hu, Koren and Volinsky, 2008): every cell gets a preference $p_{ui} = \mathbb 1[r_{ui} > 0]$ and a confidence $c_{ui} = 1 + \alpha r_{ui}$, and the two least-squares steps from class become weighted ridge regressions,
$$
x_u = \left(Y^TY + Y^T(C_u - I)Y + \lambda I\right)^{-1} Y^T C_u\, p_u ,
$$
alternating with the same update for the item factors.

### Evaluation protocol

Models are trained on one window and tested on the next. A customer is evaluated if they have at least two training baskets and buy something in the test window. For each customer we take the top 10 and report **precision@10, recall@10 and NDCG@10** (the class DCG with binary relevance, $2^{r}-1 = 1$ for a bought item), plus catalogue **coverage**. Two settings are scored:

* **all items**: anything may be recommended, including the customer's usual items;
* **new items**: subclasses the customer bought in training are removed from both the list and the ground truth. This is where CF has to *discover*.

In [ ]:
# validation window: train Nov-Dec, test Jan
val_train = dp.period(lines, end=VAL_TRAIN_END)
val_test = dp.period(lines, VAL_TRAIN_END, VAL_TEST_END)
val_ib, val_bk = dp.make_baskets(val_train)
R_val, users_val, items_val = rc.build_utility(val_train)
nb_val = val_bk.groupby("customer").size()
test_val = ev.TestSet(R_val, users_val, items_val, val_test, nb_val,
                      MIN_TRAIN_BASKETS)

density = R_val.nnz / np.prod(R_val.shape)
say(f"The validation utility matrix has **{R_val.shape[0]:,} customers × "
    f"{R_val.shape[1]:,} subclasses** with {R_val.nnz:,} non-empty cells "
    f"(density {density:.2%}). **{len(test_val.customers):,} customers** have "
    f"at least {MIN_TRAIN_BASKETS} baskets in November–December and buy "
    f"again in January; they form the validation set.")

### Tuning on November–December → January

In [ ]:
knn_grid = [rc.ItemKNN(k).fit(R_val) for k in [10, 25, 50, 100, 200]]
als_grid = [rc.ImplicitALS(d, alpha=a, seed=SEED).fit(R_val)
            for d, a in [(16, 10), (32, 10), (64, 10), (32, 3), (32, 30)]]
grid = ev.evaluate_models(knn_grid + als_grid, test_val)
grid_wide = grid.pivot(index="model", columns="mode",
                       values=["ndcg@10", "recall@10"])
grid_wide["mean ndcg"] = grid_wide["ndcg@10"].mean(axis=1)
grid_wide.sort_values("mean ndcg", ascending=False)

In [ ]:
best_name = grid_wide["mean ndcg"].idxmax()
best_val = next(m for m in knn_grid + als_grid if m.name == best_name)
best_knn = max(knn_grid, key=lambda m: grid_wide.loc[m.name, "mean ndcg"])
best_als = max(als_grid, key=lambda m: grid_wide.loc[m.name, "mean ndcg"])
print("best item-kNN:", best_knn.name)
print("best ALS     :", best_als.name)
print("selected CF model (mean NDCG over both modes):", best_name)

In [ ]:
baselines_val = [rc.Popularity().fit(R_val), rc.RepeatBuy().fit(R_val)]
val_results = ev.evaluate_models(baselines_val + [best_knn, best_als], test_val)
# with bought items removed, repeat-buy falls back to popularity: drop the copy
val_results = val_results[~((val_results["model"] == "Repeat-buy")
                            & (val_results["mode"] == "new"))]
val_results

### Profit-aware re-ranking

Each customer's top 50 candidates from the selected CF model are re-scored as

$$
s_{ui} = (1-\lambda)\, \mathrm{pct}\big(\text{relevance}_{ui}\big) + \lambda\, \mathrm{pct}\big(\text{BAP}_i\big),
$$

where both terms are rank percentiles *within the customer's 50 candidates* (1 = best), so the two scales are comparable and λ reads as "how much of the ranking is driven by value". λ = 0 is the plain recommender and λ = 1 sorts the candidates by BAP alone. As a **control**, the same is repeated with **own margin** in place of BAP, to test whether the basket adjustment adds anything over plain margin. Subclasses too rare for a BAP score get the median value.

BAP for tuning is computed on November–December only, so no January information leaks into it. Profit is measured from **actual January (later February) purchases**, never from BAP:

* **item profit**: the profit on the recommended subclasses the customer bought;
* **basket profit**: the profit of the whole baskets those subclasses were bought in, which is where a halo should show up if it is real.

λ is chosen as the value with the highest item profit whose NDCG@10 stays within 5% of λ = 0.

In [ ]:
bap_val = bm.basket_adjusted_profit(val_ib, val_bk, min_baskets=MIN_BAP_BASKETS)
values_val = {"BAP": ev.item_values(bap_val, items_val, "bap"),
              "Own margin": ev.item_values(bap_val, items_val, "own_margin")}

sweeps_val, picks_val, cand_val = {}, {}, {}
for mode, title in [("all", "All items"), ("new", "Items new to the customer")]:
    cand_val[mode] = ev.candidates(best_val, test_val, mode)
    sweeps_val[title] = ev.lambda_sweep(cand_val[mode], test_val, values_val, mode)
    picks_val[title] = ev.pick_lambda(sweeps_val[title], "BAP")

print("chosen lambda (validation):", picks_val)
plots.profit_frontier(sweeps_val, picks_val, name="06a_profit_frontier_validation");

### Does Chinese New Year distort the tuning?

January contains the New Year rush, when shoppers buy gifts and festive food they do not buy the rest of the year. If that rush drove the choice of λ, the setting would not carry over to February. We split the January ground truth into **1–13 January** (ordinary weeks) and **14–31 January** (the run-up to 24 January and the holiday) and repeat the λ sweep on each part.

In [ ]:
cny_sweeps, cny_picks = {}, {}
for title, (a, b) in {"1-13 Jan": (VAL_TRAIN_END, CNY_WINDOW_START),
                      "14-31 Jan (CNY)": (CNY_WINDOW_START, VAL_TEST_END)}.items():
    ts = ev.TestSet(R_val, users_val, items_val, dp.period(lines, a, b),
                    nb_val, MIN_TRAIN_BASKETS)
    c = ev.candidates(best_val, ts, "all")
    cny_sweeps[title] = ev.lambda_sweep(c, ts, values_val, "all")
    cny_picks[title] = ev.pick_lambda(cny_sweeps[title], "BAP")

display(pd.Series(cny_picks, name="chosen lambda").to_frame())
plots.cny_check(cny_sweeps);

In [ ]:
gap = abs(cny_picks["1-13 Jan"] - cny_picks["14-31 Jan (CNY)"])
say(f"The λ chosen on ordinary January weeks is "
    f"{cny_picks['1-13 Jan']:g} and on the New Year weeks "
    f"{cny_picks['14-31 Jan (CNY)']:g}. "
    + ("The two agree closely, so **the holiday does not distort the tuning** "
       "and the full-January value is carried to the test."
       if gap <= 0.1 else
       "They differ, so **the holiday shifts the trade-off**. To be safe, the "
       "test uses the λ chosen on the ordinary weeks, since February is not a "
       "holiday month."))
lam_choice = dict(picks_val)
if gap > 0.1:
    lam_choice["All items"] = cny_picks["1-13 Jan"]
print("lambda carried to the test:", lam_choice)

## Final Test: Train November–January, Test February

All models are refit on November–January with the settings chosen on validation, BAP is recomputed on November–January (the same `bap` table used for the store insights), and everything is scored once on February.

In [ ]:
R, users, items = rc.build_utility(train_lines)
nb_train = baskets.groupby("customer").size()
test = ev.TestSet(R, users, items, hold_lines, nb_train, MIN_TRAIN_BASKETS)

final_models = [
    rc.Popularity().fit(R),
    rc.RepeatBuy().fit(R),
    rc.ItemKNN(best_knn.k).fit(R),
    rc.ImplicitALS(best_als.factors, alpha=best_als.alpha, seed=SEED).fit(R),
]
model = next(m for m in final_models if m.name == best_name)
results = ev.evaluate_models(final_models, test)
results = results[~((results["model"] == "Repeat-buy")
                    & (results["mode"] == "new"))].reset_index(drop=True)
say(f"**{len(test.customers):,} customers** are evaluated in February "
    f"(utility matrix {R.shape[0]:,} × {R.shape[1]:,}).")
results

In [ ]:
plots.model_comparison(results);

In [ ]:
res = results.set_index(["mode", "model"])
cf_names = [m.name for m in final_models[2:]]
best_all = res.loc["all", "ndcg@10"].idxmax()
best_new = res.loc["new", "ndcg@10"].idxmax()
pop_new = res.loc[("new", "Popularity"), "ndcg@10"]
cf_new = res.loc["new"].loc[cf_names, "ndcg@10"].max()
say(f"With all items allowed, the best model is **{best_all}** "
    f"(NDCG@10 {res.loc[('all', best_all), 'ndcg@10']:.3f}). "
    + ("Grocery demand is dominated by repeat purchases, so a customer's own "
       "history is very hard to beat on accuracy. "
       if best_all == "Repeat-buy" else "")
    + f"For items new to the customer, where the customer's history cannot help, "
    f"the best model is **{best_new}** "
    f"(NDCG@10 {res.loc[('new', best_new), 'ndcg@10']:.3f}), and the best CF "
    f"model reaches {cf_new / pop_new:.2f}× the NDCG of popularity. "
    + ("This is the setting where collaborative filtering earns its keep."
       if cf_new / pop_new >= 1.1 else
       "Discovery is hard in this data: CF gains little over simply "
       "recommending best-sellers."))

### Profit-aware re-ranking on February

In [ ]:
values = {"BAP": ev.item_values(bap, items, "bap"),
          "Own margin": ev.item_values(bap, items, "own_margin")}

sweeps, cand = {}, {}
for mode, title in [("all", "All items"), ("new", "Items new to the customer")]:
    cand[mode] = ev.candidates(model, test, mode)
    sweeps[title] = ev.lambda_sweep(cand[mode], test, values, mode)

plots.profit_frontier(sweeps, lam_choice, name="06b_profit_frontier_test");

In [ ]:
rows = []
for mode, title in [("all", "All items"), ("new", "Items new to the customer")]:
    sw, lam = sweeps[title], lam_choice[title]
    for sig in ["BAP", "Own margin"]:
        r0 = sw[(sw["signal"] == sig) & (sw["lambda"] == 0)].iloc[0]
        r1 = sw[(sw["signal"] == sig) & (sw["lambda"] == lam)].iloc[0]
        rows.append({
            "setting": title, "signal": sig, "lambda": lam,
            "ndcg@10": r1["ndcg@10"],
            "ndcg change": r1["ndcg@10"] / r0["ndcg@10"] - 1,
            "item profit change": r1["profit_per_customer"]
                                  / r0["profit_per_customer"] - 1,
            "basket profit change": r1["basket_profit_per_customer"]
                                    / r0["basket_profit_per_customer"] - 1,
        })
rerank_table = pd.DataFrame(rows)
rerank_table

In [ ]:
def _row(setting, signal):
    return rerank_table[(rerank_table["setting"] == setting)
                        & (rerank_table["signal"] == signal)].iloc[0]

txt = []
for setting in ["All items", "Items new to the customer"]:
    b, m = _row(setting, "BAP"), _row(setting, "Own margin")
    txt.append(
        f"* **{setting}** (λ = {b['lambda']:g}): BAP re-ranking changes NDCG@10 "
        f"by {b['ndcg change']:+.1%}, item profit by {b['item profit change']:+.1%} "
        f"and basket profit by {b['basket profit change']:+.1%}; the margin-only "
        f"control at the same λ gives {m['item profit change']:+.1%} item profit "
        f"and {m['basket profit change']:+.1%} basket profit.")
say("\n".join(txt))

How to read the control: own margin is, by construction, the better signal for **item** profit, because item profit counts only the recommended item's own margin. BAP's case rests on **basket** profit: if the halo is real, recommending loss leaders and stars should bring in more profitable baskets even when the item itself earns little. The comparison of the two rows above is therefore the test of whether the basket adjustment adds value.

## Segments: Age Groups and Chinese New Year

Do different shoppers, or the same shoppers at different times, buy different combinations? Pair rules are mined separately in each segment (0.5% support inside the segment) and the 20 highest-lift pairs are compared with the Jaccard overlap $|A \cap B| / |A \cup B|$.

In [ ]:
age_band = baskets["age"].astype(str).map({
    "<25": "under 30", "25-29": "under 30",
    "30-34": "30-39", "35-39": "30-39",
    "40-44": "40-49", "45-49": "40-49",
    "50-54": "50+", "55-59": "50+", "60-64": "50+", ">64": "50+",
})
age_groups = {g: (age_band == g).to_numpy()
              for g in ["under 30", "30-39", "40-49", "50+"]}
age_top, age_overlap = bm.segment_rules(item_basket, baskets, age_groups,
                                        SEGMENT_SUPPORT)

cny_start = pd.Timestamp(CNY_WINDOW_START)
time_groups = {
    "Nov-Dec": (baskets["date"] < pd.Timestamp(VAL_TRAIN_END)).to_numpy(),
    "1-13 Jan": ((baskets["date"] >= pd.Timestamp(VAL_TRAIN_END))
                 & (baskets["date"] < cny_start)).to_numpy(),
    "14-31 Jan (CNY)": (baskets["date"] >= cny_start).to_numpy(),
}
time_top, time_overlap = bm.segment_rules(item_basket, baskets, time_groups,
                                          SEGMENT_SUPPORT)
plots.segment_overlap(age_overlap, "Top-20 pair overlap by age group",
                      "08a_segments_age");
plots.segment_overlap(time_overlap, "Top-20 pair overlap by period",
                      "08b_segments_time");

In [ ]:
# pairs that are top-20 in exactly one segment: what makes a segment distinct
def distinct_pairs(top, n=3):
    counts = top["pair"].value_counts()
    own = top[top["pair"].map(counts) == 1]
    return (own.sort_values("lift", ascending=False).groupby("segment").head(n)
            [["segment", "baskets", "antecedent", "consequent", "lift",
              "rel_support"]])

display(distinct_pairs(age_top))
display(distinct_pairs(time_top))

In [ ]:
off = lambda o: o.values[np.triu_indices(len(o), 1)].mean()
say(f"The average overlap of top pairs between age groups is "
    f"**{off(age_overlap):.2f}** and between periods **{off(time_overlap):.2f}**. "
    f"The lower the overlap, the more a single store-wide promotion misses: "
    f"segments with distinct top pairs are the case for targeted offers, which "
    f"is what the recommender delivers customer by customer.")

## Action List for the Store

The tables below translate the analysis into four decisions. They are generated from the results, so they update if the thresholds change.

**1. Keep these loss-making subclasses.** Loss leaders whose BAP is positive (the rest of their baskets more than pays for their own loss) and whose halo is at least two standard errors above zero.

In [ ]:
keep = lead[(lead["bap"] > 0) & (lead["halo_t"] >= 2)].sort_values("bap",
                                                                    ascending=False)
if len(pull):
    top_pull = (pull.groupby("leader")["consequent"]
                .apply(lambda s: ", ".join(s.head(3))))
    keep = keep.assign(pulls_in=top_pull.reindex(keep.index).fillna("-"))
say(f"**{len(keep):,} subclasses** qualify. Their own losses total "
    f"NT${-keep['profit'].sum():,.0f} over November–January.")
keep[["n_baskets", "own_margin", "halo", "halo_t", "bap"]
     + (["pulls_in"] if "pulls_in" in keep else [])].head(15)

**2. Review or drop these.** Dead-weight subclasses that lose money (or break even) *and* sit in baskets that earn no more than average for their size (halo at or below zero).

In [ ]:
drop = dead[dead["halo"] <= 0].sort_values("bap")
say(f"**{len(drop):,} subclasses** qualify, with combined own losses of "
    f"NT${-drop['profit'].sum():,.0f} and {drop['revenue'].sum() / bap['revenue'].sum():.2%} "
    f"of scored revenue. Their baskets already earn below the size "
    f"baseline, so the data gives no sign that repricing them to break even, "
    f"or delisting them, would cost the store basket profit.")
drop[["n_baskets", "own_margin", "margin_rate", "halo", "bap"]].head(15)

**3. Bundle or shelve together.** High-lift, profitable, common pairs from the bundle analysis above.

In [ ]:
bundles.head(10)[["a", "b", "quad_a", "quad_b", "support", "lift", "pair_margin"]]

**4. Personalised offers.** For each customer, the selected CF model's top 50 new-to-customer subclasses re-ranked with BAP at the λ chosen on validation. Five example customers are shown; the full list is produced the same way for every evaluated customer.

In [ ]:
lam_new = lam_choice["Items new to the customer"]
offers_idx = ev.rerank(cand["new"], values["BAP"], lam_new, n=5)
offers = pd.DataFrame(items.to_numpy()[offers_idx], index=test.customers,
                      columns=[f"offer {i + 1}" for i in range(5)])
quad_of = bap["quadrant"].reindex(items).fillna("unscored").to_numpy()
mix = pd.Series(quad_of[offers_idx].ravel()).value_counts(normalize=True)

say(f"Across all {len(offers):,} customers, the offers (λ = {lam_new:g}) are "
    + ", ".join(f"{share:.0%} {q.lower()}s" for q, share in mix.items())
    + ". Each customer receives a different set.")
offers.sample(5, random_state=SEED)

## Results at a Glance

In [ ]:
b_all, b_new = _row("All items", "BAP"), _row("Items new to the customer", "BAP")
glance = [
    f"* **Data**: {len(lines):,} clean lines; {n_baskets:,} training baskets; "
    f"{len(bap):,} subclasses scored.",
    f"* **Line totals**: {'confirmed' if is_total else 'not confirmed'}; "
    f"{below_cost:.1%} of clean lines sell below cost.",
    f"* **FIM**: {len(supports):,} frequent itemsets and {len(rules):,} rules "
    f"at {MIN_SUPPORT:.1%} support.",
    f"* **BAP**: {len(lead):,} loss leaders ({(lead['bap'] > 0).sum():,} with "
    f"positive BAP) and {len(dead):,} dead-weight subclasses; halo rank "
    f"correlation between training and February = {persist['halo']:.2f}.",
    f"* **Recommender (February)**: best accuracy with all items = {best_all}; "
    f"best for new items = {best_new}; selected CF model = {best_name}.",
    f"* **Profit-aware re-ranking**: all items, λ = {b_all['lambda']:g}: NDCG "
    f"{b_all['ndcg change']:+.1%}, item profit {b_all['item profit change']:+.1%}, "
    f"basket profit {b_all['basket profit change']:+.1%}. New items, "
    f"λ = {b_new['lambda']:g}: NDCG {b_new['ndcg change']:+.1%}, item profit "
    f"{b_new['item profit change']:+.1%}, basket profit "
    f"{b_new['basket profit change']:+.1%}.",
    f"* **Actions**: keep {len(keep):,} loss leaders, review {len(drop):,} "
    f"dead-weight subclasses, {min(10, len(bundles))} bundle candidates, "
    f"personalised offers for {len(offers):,} customers.",
]
say("\n".join(glance))

## Limitations

* **Association, not causation.** A large halo means a loss leader is bought alongside profitable items. It may *draw* those shoppers in, or simply be popular with shoppers who spend more anyway. Only a controlled test (removing or repricing it in some stores or weeks) can settle that.
* **Subclass level and no names.** Subclasses pool brands and pack sizes, and Ta Feng gives codes rather than names, so the store would map codes to products before acting.
* **Four months, one store.** Seasonality beyond Chinese New Year and differences across branches are out of reach.
* **Offline evaluation.** Recall and profit count purchases customers made *without* seeing our recommendations. They show that the lists contain what customers want, not how customers would react to an offer.
* **Rare items.** Subclasses in fewer than 30 baskets get no BAP and take the median value in re-ranking.

## Conclusion

Own margin alone gives a misleading picture of which products earn their place. Adjusting for the baskets a product sits in separates loss leaders worth keeping from dead weight worth reviewing, and the halo is stable enough out of time to act on. FP-Growth rules make the halo concrete by naming the profitable items each loss leader pulls in and the pairs worth bundling.

On the recommender side, the comparison against popularity and repeat-buy baselines (summarised in *Results at a Glance*) shows how much collaborative filtering adds for buy-again and for discovery. Profit-aware re-ranking lets the store choose its point on the accuracy–profit frontier with a single parameter, and comparing BAP with the margin-only control shows whether looking at the whole basket adds anything over the item alone. On this run:

In [ ]:
verdict = []
for setting in ["All items", "Items new to the customer"]:
    b, m = _row(setting, "BAP"), _row(setting, "Own margin")
    lever_item = "BAP" if b["item profit change"] > m["item profit change"] else "own margin"
    lever_basket = "BAP" if b["basket profit change"] > m["basket profit change"] else "own margin"
    verdict.append(f"* **{setting}**: {lever_item} raises item profit more; "
                   f"{lever_basket} raises basket profit more.")
adds = all(_row(s_, "BAP")["basket profit change"]
           > _row(s_, "Own margin")["basket profit change"]
           for s_ in ["All items", "Items new to the customer"])
verdict.append("\n" + ("**The basket adjustment adds value**: BAP-based offers bring in "
                       "more profitable baskets than margin-based offers at the same accuracy cost."
                       if adds else
                       "**The basket adjustment does not consistently beat plain margin** "
                       "in this offline test; its main value is diagnostic (which "
                       "loss-makers to keep), not in ranking offers."))
say("\n".join(verdict))

**Recommendations:** keep the listed loss leaders and stock them near the items they pull in; reprice or delist the dead-weight subclasses; trial the top bundles; send each customer the re-ranked offers; and confirm the largest loss-leader effects with a controlled price test before making permanent changes.

# References

* C. Aggarwal, "Data Mining: The Textbook", Springer, 2015.
* C. Aggarwal, "Recommender Systems: The Textbook", Springer, 2016.
* J. Leskovec, A. Rajaraman and J. Ullman, "Mining of Massive Datasets", 3rd ed., Cambridge University Press, 2020.
* C. Borgelt, "Frequent item set mining", WIREs Data Mining and Knowledge Discovery, 2(6), 437–456, 2012 (pyFIM).
* Y. Hu, Y. Koren and C. Volinsky, "Collaborative Filtering for Implicit Feedback Datasets", Proc. IEEE ICDM, 263–272, 2008.
* M. Deshpande and G. Karypis, "Item-based top-N recommendation algorithms", ACM Transactions on Information Systems, 22(1), 143–177, 2004.
* C.-N. Hsu, H.-H. Chung and H.-S. Huang, "Mining skewed and sparse transaction data for personalized shopping recommendation", Machine Learning, 57, 35–59, 2004 (Ta Feng dataset).